# 08 — Batch Mode & Multi-Device Synchronization

This notebook covers advanced patterns for coordinating multiple Ivium devices simultaneously:

- **Status parameter** — a global integer shared between Python and IviumSoft used as a synchronization signal
- **Multi-device setpoints** — apply potential or current to multiple devices in one call without switching instances
- **Parallel experiment patterns** — start and monitor measurements on all devices concurrently

### Typical use cases

- Trigger all devices to start a measurement at the same time
- Have IviumSoft-side scripts wait for a Python signal before proceeding
- Apply the same setpoint to multiple devices simultaneously
- Run independent experiments in parallel and collect all results

### Prerequisites
- Multiple IviumSoft instances running (or one Ivium-n-Soft with multiple channels)
- At least two devices connected for multi-device sections
- Error handling reference: `01_getting_started.ipynb` — Section 4

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
print(f"Active instances: {Pyvium.get_active_iviumsoft_instances()}")

## Multichannel vs MC Mode — Choosing the Right Architecture

IviumSoft provides two distinct multi-device modes. Choose based on the number of channels:

| Feature | **Multichannel** | **MC Mode** |
|---------|-----------------|-------------|
| Max channels | **32** | **128** |
| Real-time display | Yes — live current/potential traces | No |
| Data access during run | `get_available_data_points_number()` works | Not accessible until complete |
| IviumSoft control | `select_channel(n)` (within one window) | Separate IviumSoft instances or scripted |
| Use case | ≤32 electrodes, need live monitoring | >32 electrodes, throughput matters more |
| Pyvium API | `select_channel(n)` per channel | `select_iviumsoft_instance(n)` per instance |

> **Rule of thumb:** For ≤32 channels use **Multichannel** (Ivium-n-Soft). For >32 channels use **MC Mode** with multiple IviumSoft instances.

> **Note:** Multichannel mode uses `select_channel(n)` to switch between channels within a **single** IviumSoft window. MC Mode uses `select_iviumsoft_instance(n)` to address separate IviumSoft windows.

## 1. The Status Parameter

`set_status_par(value)` / `get_status_par()` expose a global integer shared between Python and IviumSoft. It is per-instance — each IviumSoft window has its own status parameter.

**Key facts from the IviumSoft manual:**
- StatusPar is **initialized to `-1` at PC startup**
- It **retains its last-set value** until the PC is restarted — it does not reset between measurements or driver open/close cycles
- Both Python (via pyvium) and IviumSoft Batch scripts can read and write it

### Synchronization convention

Choose any convention that fits your workflow. A common one:

| Value | Who sets it | Meaning |
|-------|------------|---------|
| `-1` | PC startup default | Uninitialized |
| `0` | Python | Reset / idle |
| `1` | Python | "Ready — IviumSoft may start" |
| `2` | IviumSoft Batch | "Measurement running" |
| `3` | IviumSoft Batch | "Measurement done" |
| `-99` | Either | Abort / error |

In [ ]:
Pyvium.select_iviumsoft_instance(1)

# Write a value
Pyvium.set_status_par(0)
print(f"Set status par to 0")

# Read it back
val = Pyvium.get_status_par()
print(f"Get status par: {val}")

## 2. Python → IviumSoft Trigger

Pattern: Python sets the status parameter to `1` to signal IviumSoft to start a method. IviumSoft-side scripting polls the status parameter and begins when it sees `1`.

This lets IviumSoft-side automation react to Python events.

In [ ]:
# Signal IviumSoft to proceed
Pyvium.select_iviumsoft_instance(1)
Pyvium.set_status_par(1)  # "go"
print("Triggered IviumSoft instance 1 (status par = 1)")

## 3. The Full Bidirectional Contract

The status parameter works in **both directions**. IviumSoft Batch scripts have two matching DirectCommands:

| Direction | Python side | IviumSoft Batch side |
|-----------|-------------|----------------------|
| Python → IviumSoft | `set_status_par(1)` | `WaitForStatusPar` — halts Batch until value matches |
| IviumSoft → Python | `get_status_par()` in a poll loop | `SetStatusPar` — Batch sets the value when done |

### IviumSoft Batch script configuration (for reference)

To make an IviumSoft Batch script wait for Python's go-signal and then report back when done, configure a `DirectCommand` line in the Batch Editor with:

```
DirectCommand:
  WaitForStatusPar  ✓  Value = 1   Timeout = 300 s   ← waits for Python to say "go"
  (then run your methods)
  SetStatusPar      ✓  Value = 3                      ← signals Python "done"
```

Python then drives the sequence:

```python
Pyvium.set_status_par(0)   # reset
# ... prepare experiment ...
Pyvium.set_status_par(1)   # signal IviumSoft to start
# ... wait for IviumSoft to set status = 3 ...
```

In [ ]:
def wait_for_status(instance: int, target_value: int, timeout_s: float = 120.0):
    """Block until the given instance's status par reaches target_value, or timeout."""
    Pyvium.select_iviumsoft_instance(instance)
    t_start = time.time()

    while True:
        current = Pyvium.get_status_par()
        if current == target_value:
            return True
        if time.time() - t_start > timeout_s:
            print(f"Timeout waiting for instance {instance} status={target_value}")
            return False
        time.sleep(0.2)

# Example: wait up to 60 s for IviumSoft to set status par to 3 ("done")
# done = wait_for_status(instance=1, target_value=3, timeout_s=60.0)
# if done:
#     print("Measurement done")
print("wait_for_status() helper defined")

## 4. Alternative Triggers — Digital Output and Analog Input

Besides StatusPar, IviumSoft Batch Mode supports two hardware-level trigger mechanisms that Python can drive directly:

### Digital trigger (`WaitForDigIn1`)

IviumSoft Batch can halt on a `DirectCommand` line configured as:
```
WaitForDigIn1  ✓   WaitForHi = ✓   Timeout = 60 s
```
Python drives this by asserting or releasing a digital output line on the external port, which is wired back to the device's digital input 1.

### Analog trigger (`WaitForAn1`)

IviumSoft Batch can halt until Analog Input 1 exceeds a threshold voltage:
```
WaitForAn1  ✓   UntilAn1 > 2.5 V   Timeout = 60 s
```
Python drives this by setting DAC channel 0 to the trigger voltage.

Both mechanisms have a **timeout** — the Batch script continues regardless once the timeout expires, so experiments are never permanently stalled by a failed Python signal.

In [ ]:
import time

# --- Digital trigger pattern ---
# IviumSoft Batch is configured with WaitForDigIn1 (WaitForHi, Timeout=60s)
# Python asserts digital output line 0 to release the Batch hold

def trigger_batch_via_digital(line: int = 0):
    """Assert a digital output line to trigger an IviumSoft Batch WaitForDigIn."""
    Pyvium.set_digital_output(1 << line)   # drive line HIGH
    print(f"Digital line {line} HIGH — IviumSoft Batch trigger sent")

def release_digital_trigger(line: int = 0):
    """Release the digital line after the Batch has seen the trigger."""
    current = Pyvium.get_digital_input()
    Pyvium.set_digital_output(0)            # all lines LOW
    print("Digital lines reset to LOW")

# Example usage:
# trigger_batch_via_digital(0)
# time.sleep(0.1)
# release_digital_trigger(0)

print("Digital trigger helpers defined")

# --- Analog trigger pattern ---
# IviumSoft Batch is configured with WaitForAn1 (UntilAn1 > 2.5 V, Timeout=60s)
# Python raises DAC 0 above the threshold to release the hold

TRIGGER_VOLTAGE  = 3.0   # V — must exceed the Batch threshold
IDLE_VOLTAGE     = 0.0   # V

def trigger_batch_via_analog():
    Pyvium.set_dac(0, TRIGGER_VOLTAGE)
    print(f"DAC 0 → {TRIGGER_VOLTAGE} V — IviumSoft Batch analog trigger sent")

def release_analog_trigger():
    Pyvium.set_dac(0, IDLE_VOLTAGE)
    print(f"DAC 0 → {IDLE_VOLTAGE} V (idle)")

# Example usage:
# trigger_batch_via_analog()
# time.sleep(0.5)
# release_analog_trigger()

print("Analog trigger helpers defined")

## 5. Launching Python from an IviumSoft Batch Script (`ExecuteProgram`)

IviumSoft Batch Mode has an `ExecuteProgram` DirectCommand that can launch any external executable — including a Python script — at a specific point in a Batch sequence. With `WaitUntilFinished = ✓`, the Batch pauses until the Python process exits.

**Batch configuration:**
```
DirectCommand:
  ExecuteProgram     ✓
  Program Name:      python.exe
  Path:              C:\Users\...\Scripts\
  Command Line:      C:\path\to\my_analysis.py --output C:\data\result.csv
  WaitUntilFinished: ✓
```

**Use cases:**
- Post-process data immediately after each scan within a Batch loop
- Send a notification (email, Slack) when a long experiment finishes
- Read the measurement result and write a new parameter file that the next Batch line picks up

**Flow from IviumSoft's perspective:**
```
[Batch line N]  ExecuteMethod           ← run experiment
[Batch line N+1] ExecuteProgram         ← launch Python, wait for exit
[Batch line N+2] Loop back / next scan  ← Python result already on disk
```

This is the inverse of the patterns above: instead of Python driving IviumSoft, IviumSoft drives Python at precisely the right moment in the sequence.

## IviumSoft Batch — EditMethod and SI Units

When an IviumSoft Batch script modifies method parameters using the **`EditMethod`** DirectCommand, all parameter values must be specified in **SI base units** — regardless of the display units shown in the IviumSoft UI.

This is important when Python generates parameter values that are subsequently read by an IviumSoft Batch script (e.g., written to a file that Batch picks up), or when you are debugging why a Batch-modified method does not behave as expected.

| Parameter | Display unit in IviumSoft | SI unit for EditMethod / Pyvium |
|-----------|--------------------------|--------------------------------|
| Potential | mV | **V** |
| Current | µA, mA | **A** |
| Time | ms, min, h | **s** |
| Frequency | kHz | **Hz** |
| Scan rate | mV/s | **V/s** |

> **This is the same convention used by Pyvium's `set_method_parameter()`** — all values are in SI base units. IviumSoft handles unit display conversion internally.

> **Gotcha:** If you set `'scanrate'` to `'50'` expecting 50 mV/s, the actual scan rate will be 50 V/s. Use `'0.05'` for 50 mV/s.

## 4. Triggering All Instances Simultaneously

By quickly looping through all instances and setting the status parameter, you can trigger all devices near-simultaneously.

In [ ]:
def broadcast_status(value: int):
    """Set the status parameter on all active IviumSoft instances."""
    instances = Pyvium.get_active_iviumsoft_instances()
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.set_status_par(value)
    print(f"Broadcast status={value} to instances {instances}")

broadcast_status(0)   # reset all to idle
time.sleep(1.0)
broadcast_status(1)   # trigger all

## 5. Multi-Device Setpoints

`set_device_potential(instance, V)` and `set_device_current(instance, A)` apply a setpoint to a specific device instance **without changing the currently selected instance**. This allows rapid setpoint updates across all devices without the overhead of instance-switching.

- `instance`: the IviumSoft instance number (1-based)
- Both functions require the driver to be open but **not** the target device to be the currently selected instance

In [ ]:
# Apply the same potential to all active instances simultaneously
HOLD_POTENTIAL = 0.1  # V

instances = Pyvium.get_active_iviumsoft_instances()
for inst in instances:
    Pyvium.set_device_potential(inst, HOLD_POTENTIAL)
    print(f"Instance {inst}: potential → {HOLD_POTENTIAL} V")

print("All devices set")

In [ ]:
# Apply a gradient of potentials across devices
potentials = [0.0, 0.1, 0.2, 0.3]  # V — one per instance

for inst, V in zip(instances, potentials):
    Pyvium.set_device_potential(inst, V)
    print(f"Instance {inst}: {V:.2f} V")

## 6. Running Methods on All Devices in Parallel

Start a method on each device, then collect data from all of them after all complete.

In [ ]:
METHOD_PATH     = r"C:/IviumStat/datafiles/TEST1.imf"
EXPECTED_POINTS = 160
POLL_INTERVAL   = 0.5

def start_all(instances, method_path):
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.connect_device()
        Pyvium.start_method(method_path)
        print(f"Instance {inst}: method started")

def wait_all(instances, expected_points):
    remaining = set(instances)
    while remaining:
        finished = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            n = Pyvium.get_available_data_points_number()
            if n >= expected_points:
                finished.add(inst)
        for inst in finished:
            print(f"  Instance {inst}: complete")
        remaining -= finished
        if remaining:
            time.sleep(POLL_INTERVAL)
    print("All measurements complete")

def collect_all(instances):
    results = {}
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        n = Pyvium.get_available_data_points_number()
        rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]
        results[inst] = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])
        print(f"  Instance {inst}: {len(rows)} points collected")
    return results

print("Helpers defined. Uncomment below to run:")
# start_all(instances, METHOD_PATH)
# wait_all(instances, EXPECTED_POINTS)
# data = collect_all(instances)

## 7. Plotting Multi-Device Results

In [ ]:
# Assumes `data` dict was populated by collect_all() above
# data = collect_all(instances)

# Synthetic example for illustration
import numpy as np
data = {
    1: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 1e-5}),
    2: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 2e-5}),
}

fig, ax = plt.subplots(figsize=(8, 5))
for inst, df in data.items():
    ax.plot(df["E (V)"], df["I (A)"] * 1e6, label=f"Device {inst}")

ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title("Parallel LSV — All Devices")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Complete Synchronized Workflow

Full template: connect all devices, run a method in parallel, collect data, save and plot.

## Synchronized Channels — Master-Last Rule

When running multiple channels in synchronized mode (using IviumSoft's channel synchronization feature), the **master channel must always be started last**. The master channel acts as the clock source for all synchronized channels; slave channels wait in a ready state until the master starts.

**Critical rule:** If you start the master first, the synchronization signal fires before any slave channel is listening — all slaves miss the trigger and start independently (unsynchronized).

```python
# Correct order for synchronized multi-channel start
slaves  = [2, 3, 4]  # slave channel instances
master  = 1          # master channel instance

# 1. Start all slaves first (they enter wait state)
for inst in slaves:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(method_path)
    print(f"Slave instance {inst}: waiting for sync")

# 2. Small delay to ensure slaves are ready
time.sleep(0.5)

# 3. Start master LAST — fires the synchronization trigger
Pyvium.select_iviumsoft_instance(master)
Pyvium.start_method(method_path)
print(f"Master instance {master}: started — sync trigger fired")
```

> This rule applies to IviumSoft's synchronized channel feature (configured in the Multichannel setup). It is different from the software-level status parameter synchronization described in the sections above, which is not timing-critical.

In [ ]:
import os

METHOD_PATH    = r"C:/IviumStat/datafiles/TEST1.imf"
OUTPUT_DIR     = r"C:/IviumStat/Data"
EXPECTED_PTS   = 160

Pyvium.open_driver()
instances = Pyvium.get_active_iviumsoft_instances()
print(f"Devices: {instances}")

# Step 1: connect all and start
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.connect_device()

for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(METHOD_PATH)
    print(f"  Instance {inst}: started")

# Step 2: wait for all to finish
remaining = set(instances)
while remaining:
    done = set()
    for inst in remaining:
        Pyvium.select_iviumsoft_instance(inst)
        if Pyvium.get_available_data_points_number() >= EXPECTED_PTS:
            done.add(inst)
    remaining -= done
    if remaining:
        time.sleep(0.5)
print("All done")

# Step 3: collect and save
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    out = os.path.join(OUTPUT_DIR, f"device_{inst}.idf")
    if os.path.isdir(OUTPUT_DIR):
        Pyvium.save_data(out)
        print(f"  Instance {inst} → {out}")

Pyvium.close_driver()

---

## Summary

### Status parameter (synchronization)

| Task | Python | IviumSoft Batch side |
|------|--------|----------------------|
| Write status | `set_status_par(value)` | `SetStatusPar` DirectCommand |
| Read / wait on status | `get_status_par()` in poll loop | `WaitForStatusPar` DirectCommand |
| Broadcast to all instances | Loop + `select_iviumsoft_instance(n)` + `set_status_par(v)` | — |
| Initial value at PC startup | — | **-1** (retains last value until PC reset) |

### Alternative hardware triggers

| Trigger type | Python sends | IviumSoft Batch waits on |
|-------------|-------------|--------------------------|
| Digital | `set_digital_output(bitmask)` | `WaitForDigIn1` (HI or LO, with timeout) |
| Analog | `set_dac(0, voltage)` | `WaitForAn1 > threshold` (with timeout) |
| Launch Python | — | `ExecuteProgram` (optional `WaitUntilFinished`) |

### Multi-device setpoints (no instance switch needed)

| Task | Method |
|------|--------|
| Set potential on any instance | `set_device_potential(instance, V)` |
| Set current on any instance | `set_device_current(instance, A)` |

### Multichannel architecture choice

| Channels | Architecture | Pyvium API |
|----------|-------------|------------|
| ≤ 32 | Multichannel (Ivium-n-Soft) — live display | `select_channel(n)` |
| > 32 | MC Mode — no live display | `select_iviumsoft_instance(n)` |

### Synchronized channels rule

**Always start slave channels before the master channel.** The master fires the sync trigger on start — slaves must already be waiting.

### EditMethod / set_method_parameter units

All parameter values are in **SI base units** (V, A, s, Hz) regardless of the display units shown in the IviumSoft UI.

### Parallel measurement pattern

```python
# 1. Start all
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(path)

# 2. Wait for all
while not all_done(instances):
    time.sleep(0.5)

# 3. Collect from all
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    data[inst] = read_all_points()
```

---

## Notebook Series Complete

| Notebook | Topic | Hardware needed? |
|----------|-------|:-:|
| 01 | Getting started, error handling | No (just IviumSoft) |
| 02 | Device & instance management | Yes |
| 03 | Direct mode — potentiostat basics | Yes |
| 04 | Direct mode — traces, DAC/ADC, digital I/O, AC | Yes |
| 05 | BiStat (WE2) and WE32 multichannel | Yes (+ accessories) |
| 06 | Method mode — run experiments, read data | Yes |
| 07 | IDF file parsing and CSV export | **No** |
| 08 | Batch mode & multi-device synchronization | Yes (multiple devices) |